# Load pretrained weights into your own model
Works for any model you've coded. Practice run: GPT-2 → your `GPT2Model`.

```
1 Download    the model's .safetensors file
2 Theirs      print names + shapes
3 Yours       sizes from config.json · print names + shapes
4 Map + copy  match by name and shape · fix: copy · flip · split · reuse · skip
5 Test        output must match the original model
```
✍️ = your turn · solution at the bottom

## 0 · Setup
Installs `safetensors` (the weight-file format) and loads your model code.

In [ ]:
%%capture
%pip install safetensors
%run GPT2Model.ipynb

## 1 · Download
`w` is a dict: name → numbers. Other model: change the link.
Big models come in several files: load each and merge, `w = {**load_file(a), **load_file(b)}`.

In [ ]:
import os, urllib.request
from safetensors.torch import load_file

URL = "https://huggingface.co/openai-community/gpt2/resolve/main/model.safetensors"
if not os.path.exists("gpt2.safetensors"):
    urllib.request.urlretrieve(URL, "gpt2.safetensors")  # ~520 MB, once
w = load_file("gpt2.safetensors")

## 2 · Theirs
Block 0 + the rest (other blocks repeat).

In [ ]:
import re

def block0(name):  # no block number, or block 0
    m = re.search(r"\.(\d+)\.", name)
    return m is None or m.group(1) == "0"

def show(d):
    for k, v in sorted(d.items()):
        if block0(k):
            print(f"{k:36} {list(v.shape)}")

show(w)

## 3 · Yours
Your sizes must match theirs: they're in `config.json`, next to the weights
(GPT-2: `vocab_size 50257 · n_positions 1024 · n_embd 768 · n_layer 12 · n_head 12`).
Theirs also has `c_attn.bias`, so `qkv_bias=True`.

In [ ]:
model = GPT2Model({**GPT_CONFIG_124M, "qkv_bias": True, "drop_rate": 0.0})
show(model.state_dict())

## 4 · Map + copy ✍️
Match each pair from steps 2–3 by similar words and shapes, then one `assign` per pair.

| Shapes | Fix |
|---|---|
| same | copy |
| reversed | flip: `.T` |
| theirs = 3 × yours | split: `.chunk(3, dim=-1)` |
| yours has no match | reuse one (GPT-2: `out_head` = `wte`) |
| theirs has no match | skip |

In [ ]:
import torch

done = set()
def assign(slot, numbers):  # copies, refuses wrong shapes, remembers what's filled
    assert slot.shape == numbers.shape, f"{list(slot.shape)} vs {list(numbers.shape)}"
    with torch.no_grad():
        slot.copy_(numbers)
    done.add(id(slot))

assign(model.tok_emb.weight, w["wte.weight"])                    # copy
for i, b in enumerate(model.trf_blocks):
    q, k, v = w[f"h.{i}.attn.c_attn.weight"].chunk(3, dim=-1)    # split
    assign(b.attn.W_query.weight, q.T)                           # flip
assign(model.out_head.weight, w["wte.weight"])                   # reuse

# ✍️ add the rest; step 5 lists what's missing

## 5 · Test
Lists what's not filled yet (block 0 shown), then generates. Correct weights give:
*"Every effort moves you forward. The first step is to understand the importance of your work"*

In [ ]:
import tiktoken

missing = [n for n, p in model.named_parameters() if id(p) not in done]
print(f"Not filled: {len(missing)}", [n for n in missing if block0(n)] if missing else "✅")
tok = tiktoken.get_encoding("gpt2")
ids = torch.tensor([tok.encode("Every effort moves you")])
model.eval()
with torch.no_grad():
    for _ in range(15):
        ids = torch.cat([ids, model(ids)[:, -1].argmax(-1, keepdim=True)], dim=1)
print(tok.decode(ids[0].tolist()))

## Solution (try step 4 first)
Then run step 5 again.

In [ ]:
assign(model.tok_emb.weight, w["wte.weight"])
assign(model.pos_emb.weight, w["wpe.weight"])
for i, b in enumerate(model.trf_blocks):
    h = f"h.{i}."
    q, k, v = w[h + "attn.c_attn.weight"].chunk(3, dim=-1)
    assign(b.attn.W_query.weight, q.T); assign(b.attn.W_key.weight, k.T); assign(b.attn.W_value.weight, v.T)
    q, k, v = w[h + "attn.c_attn.bias"].chunk(3, dim=-1)
    assign(b.attn.W_query.bias, q); assign(b.attn.W_key.bias, k); assign(b.attn.W_value.bias, v)
    assign(b.attn.out_proj.weight, w[h + "attn.c_proj.weight"].T)
    assign(b.attn.out_proj.bias, w[h + "attn.c_proj.bias"])
    assign(b.ff.layers[0].weight, w[h + "mlp.c_fc.weight"].T)
    assign(b.ff.layers[0].bias, w[h + "mlp.c_fc.bias"])
    assign(b.ff.layers[2].weight, w[h + "mlp.c_proj.weight"].T)
    assign(b.ff.layers[2].bias, w[h + "mlp.c_proj.bias"])
    assign(b.norm1.scale, w[h + "ln_1.weight"]); assign(b.norm1.shift, w[h + "ln_1.bias"])
    assign(b.norm2.scale, w[h + "ln_2.weight"]); assign(b.norm2.shift, w[h + "ln_2.bias"])
assign(model.final_norm.scale, w["ln_f.weight"])
assign(model.final_norm.shift, w["ln_f.bias"])
assign(model.out_head.weight, w["wte.weight"])  # reuse: GPT-2 has no separate output layer